# How to use this notebook

1. Change the `RUN_DIR` value in the first code cell only.
2. Run all cells from top to bottom.
3. The notebook validates that the standard output files exist before reading them.
4. The notebook loads saved outputs only. It does **not** rerun MILPs, benchmarks, or backtests.
5. Gamma values here are fixed risk-preference policies, not test-week tuning candidates.


In [ ]:
# Input cell: change RUN_DIR only
RUN_DIR = r"scripts/Data/03_Hydrogen_Test_Case/runs/20260519_184156_phase_e2_all_selected_test_weeks_cvar_policy_three_model"


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

run_dir_input = Path(RUN_DIR)
if run_dir_input.is_absolute() and run_dir_input.exists():
    run_dir = run_dir_input.resolve()
else:
    run_dir = None
    for base_dir in [Path.cwd(), *Path.cwd().parents]:
        candidate = (base_dir / run_dir_input).resolve()
        if candidate.exists():
            run_dir = candidate
            break
    if run_dir is None:
        run_dir = (Path.cwd() / run_dir_input).resolve()
NOTEBOOK_INPUTS = run_dir / 'notebook_inputs'
FIGURES_DIR = run_dir / 'figures'
required_files = [
    NOTEBOOK_INPUTS / 'support_days.csv',
    NOTEBOOK_INPUTS / 'daily_metrics.csv',
    NOTEBOOK_INPUTS / 'weekly_metrics.csv',
    NOTEBOOK_INPUTS / 'benchmark_metrics.csv',
    NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv',
    NOTEBOOK_INPUTS / 'cvar_bid_firmness_metrics.csv',
    NOTEBOOK_INPUTS / 'cvar_validation_checks.csv',
    NOTEBOOK_INPUTS / 'validation_checks_all_runs.csv',
    FIGURES_DIR / 'fig_economic_decomposition_by_model_gamma.png',
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError('Missing required run outputs:\n' + '\n'.join(missing_files))
selected_weeks_path = NOTEBOOK_INPUTS / 'selected_weeks_manifest_table.csv'
selected_week_path = NOTEBOOK_INPUTS / 'selected_week_manifest_table.csv'
if selected_weeks_path.exists():
    selected_weeks = pd.read_csv(selected_weeks_path)
elif selected_week_path.exists():
    selected_weeks = pd.read_csv(selected_week_path)
else:
    raise FileNotFoundError('Expected selected_weeks_manifest_table.csv or selected_week_manifest_table.csv in notebook_inputs.')
frontier_path = NOTEBOOK_INPUTS / 'cvar_frontier_aggregated.csv'
frontier_week_path = NOTEBOOK_INPUTS / 'cvar_frontier_by_model_week.csv'
frontier_model_path = NOTEBOOK_INPUTS / 'cvar_frontier_by_model.csv'
if frontier_path.exists():
    frontier_aggregated = pd.read_csv(frontier_path)
else:
    frontier_aggregated = pd.DataFrame()
if frontier_week_path.exists():
    frontier_weekly = pd.read_csv(frontier_week_path)
elif frontier_model_path.exists():
    frontier_weekly = pd.read_csv(frontier_model_path)
else:
    frontier_weekly = pd.DataFrame()
daily_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'daily_metrics.csv')
weekly_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'weekly_metrics.csv')
benchmark_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'benchmark_metrics.csv')
perfect_foresight_metrics = pd.read_csv(NOTEBOOK_INPUTS / 'perfect_foresight_metrics.csv')
bid_firmness = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_bid_firmness_metrics.csv')
cvar_checks = pd.read_csv(NOTEBOOK_INPUTS / 'cvar_validation_checks.csv')
validation_checks = pd.read_csv(NOTEBOOK_INPUTS / 'validation_checks_all_runs.csv')
model_selection_path = NOTEBOOK_INPUTS / 'model_selection_evidence_summary.csv'
model_selection = pd.read_csv(model_selection_path) if model_selection_path.exists() else pd.DataFrame()
support_days = pd.read_csv(NOTEBOOK_INPUTS / 'support_days.csv')
is_multi_week = int(selected_weeks['week_label'].astype(str).nunique()) > 1


## Run scope and validation


In [ ]:
hard_fail_count = int(validation_checks.loc[(validation_checks['severity'].astype(str) == 'hard_fail') & (validation_checks['status'].astype(str) == 'fail')].shape[0])
cvar_fail_count = int(cvar_checks.loc[cvar_checks['status'].astype(str) == 'fail'].shape[0])
summary = pd.DataFrame([{
    'week_count': int(selected_weeks['week_label'].astype(str).nunique()),
    'weeks': ', '.join(selected_weeks['week_label'].astype(str).tolist()),
    'models': ', '.join(weekly_metrics['model_label'].astype(str).drop_duplicates().tolist()),
    'gamma_values': ', '.join([f'{value:.2f}' for value in sorted(weekly_metrics['cvar_gamma'].astype(float).unique().tolist())]),
    'alpha': ', '.join([f'{value:.2f}' for value in sorted(weekly_metrics['cvar_alpha'].astype(float).unique().tolist())]),
    'price_insensitive_benchmark': not benchmark_metrics.empty,
    'perfect_foresight_benchmark': not perfect_foresight_metrics.empty,
    'hard_validation_failures': hard_fail_count,
    'cvar_failures': cvar_fail_count,
}])
display(summary)
display(selected_weeks[['week_label', 'delivery_start_date', 'delivery_end_date', 'selection_reason']])


## Headline aggregated performance table


In [ ]:
if not frontier_aggregated.empty:
    display(frontier_aggregated.round(3))
elif not model_selection.empty:
    display(model_selection.loc[model_selection['row_type'].astype(str).eq('model_gamma')].round(3))
else:
    agg = weekly_metrics.groupby(['model_label', 'cvar_gamma'], as_index=False).agg(realised_adjusted_profit=('realised_adjusted_profit','mean'), cvar_tail_profit=('cvar_tail_profit','mean'), value_captured_vs_perfect_foresight=('value_captured_vs_perfect_foresight','mean'))
    display(agg.round(3))


## Weekly performance table


In [ ]:
display(weekly_metrics[['week_label','model_label','cvar_gamma','realised_adjusted_profit','stochastic_minus_benchmark_profit','perfect_foresight_profit','value_captured_vs_perfect_foresight','cvar_tail_profit','worst_scenario_profit','clearing_ratio','rejected_energy_mwh','unused_cleared_energy_mwh','hydrogen_sold_or_compressed_kg','shortfall_kg','average_actual_price_paid','solve_time_seconds']].sort_values(['week_label','model_label','cvar_gamma']).round(3))


## Risk-return figures


In [ ]:
for figure_name in [
    'fig_aggregated_risk_return_profit_vs_cvar_tail_profit.png',
    'fig_aggregated_value_captured_vs_profit.png',
    'fig_gamma_vs_realised_profit_by_model.png',
    'fig_gamma_vs_cvar_tail_profit_by_model.png',
    'fig_gamma_vs_worst_scenario_profit_by_model.png',
]:
    path = FIGURES_DIR / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))


## Benchmark comparison


In [ ]:
display(benchmark_metrics.round(3))
display(perfect_foresight_metrics.round(3))


In [ ]:
for figure_name in [
    'fig_weekly_profit_vs_benchmarks.png',
    'fig_value_captured_vs_perfect_foresight.png',
]:
    path = FIGURES_DIR / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))


## Bidding and clearing behaviour


In [ ]:
display(bid_firmness.round(3))
for figure_name in [
    'fig_clearing_ratio_by_model_gamma_week.png',
    'fig_rejected_energy_by_model_gamma_week.png',
    'fig_unused_energy_by_model_gamma_week.png',
    'fig_bid_firmness_by_model_gamma.png',
]:
    path = FIGURES_DIR / figure_name
    if path.exists():
        display(Markdown(f'### {figure_name}'))
        display(Image(filename=str(path)))


## Scenario and price context


In [ ]:
scenario_figures = sorted(FIGURES_DIR.glob('fig_scenario_fan_*.png'))
for path in scenario_figures:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))


## Operational asset behaviour


In [ ]:
example_figures = sorted(FIGURES_DIR.glob('fig_example_day_operation_*.png'))
for path in example_figures:
    display(Markdown(f'### {path.name}'))
    display(Image(filename=str(path)))


## Economic decomposition


In [ ]:
path = FIGURES_DIR / 'fig_economic_decomposition_by_model_gamma.png'
if path.exists():
    display(Image(filename=str(path)))


## Model-selection evidence


In [ ]:
if not model_selection.empty:
    display(model_selection.round(3))
path = FIGURES_DIR / 'fig_model_selection_summary.png'
if path.exists():
    display(Image(filename=str(path)))


## Interpretation helper


In [ ]:
best_realised = weekly_metrics.loc[pd.to_numeric(weekly_metrics['realised_adjusted_profit'], errors='coerce').idxmax()]
best_capture = weekly_metrics.loc[pd.to_numeric(weekly_metrics['value_captured_vs_perfect_foresight'], errors='coerce').idxmax()]
lowest_shortfall = weekly_metrics.loc[pd.to_numeric(weekly_metrics['shortfall_kg'], errors='coerce').idxmin()]
lowest_rejected = weekly_metrics.loc[pd.to_numeric(weekly_metrics['rejected_energy_mwh'], errors='coerce').idxmin()]
best_tail = weekly_metrics.loc[pd.to_numeric(weekly_metrics['cvar_tail_profit'], errors='coerce').idxmax()]
bullet_lines = [
    f"- best realised profit: {best_realised['model_label']} gamma={float(best_realised['cvar_gamma']):.2f} {best_realised['week_label']} ({float(best_realised['realised_adjusted_profit']):,.0f} EUR)",
    f"- best value captured vs perfect foresight: {best_capture['model_label']} gamma={float(best_capture['cvar_gamma']):.2f} {best_capture['week_label']} ({float(best_capture['value_captured_vs_perfect_foresight']):.3f})",
    f"- lowest shortfall: {lowest_shortfall['model_label']} gamma={float(lowest_shortfall['cvar_gamma']):.2f} ({float(lowest_shortfall['shortfall_kg']):,.1f} kg)",
    f"- lowest rejected energy: {lowest_rejected['model_label']} gamma={float(lowest_rejected['cvar_gamma']):.2f} ({float(lowest_rejected['rejected_energy_mwh']):,.1f} MWh)",
    f"- best CVaR tail profit: {best_tail['model_label']} gamma={float(best_tail['cvar_gamma']):.2f} {best_tail['week_label']} ({float(best_tail['cvar_tail_profit']):,.0f} EUR)",
    f"- hard validation failures: {hard_fail_count}",
    f"- CVaR check failures: {cvar_fail_count}",
]
display(Markdown('\n'.join(bullet_lines)))
